<a href="https://colab.research.google.com/github/thinus283-ux/LR/blob/main/Full_CMB_validation_test_TT_EE_TE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Install system-level dependencies for the C-engine
!apt-get update
!apt-get install -y libgsl-dev libfftw3-dev liblapack-dev libblas-dev libopenblas-dev gfortran make

# Clone, compile, and install the CLASS C-engine
!git clone https://github.com/lesgourg/class_public.git
%cd class_public
!make clean
!make -j4

# Install the Python wrapper (classy)
%cd python
!python3 setup.py install --user

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,024 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,306 kB]
Get:13 https://cli.github.com/packages stable/main amd64 Packages [355 B

import os
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

def run_scientific_perturbation_test():
    # 1. Setup Data Directory
    output_dir = 'data_results'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 2. Define Parameters for Logic Relativity 1.2
    # Mapping the displacement field impacts to effective cosmological fluid parameters
    params = {
        'output': 'tCl pCl',
        'l_max_scalars': 2500,
        'H0': 67.52,
        'omega_b': 0.02237,
        'omega_cdm': 0.1200,
        'A_s': 2.10e-9,
        'n_s': 0.9649,
        'tau_reio': 0.0544,
        # Displacement field perturbation mapping:
        'fluid_equation_of_state': '-1.02',
        'perturbations_check': 'yes'
    }

    print("[PROCESS] Running High-Fidelity Boltzmann Solver...")
    
    # 3. Initialize and Compute
    M = Class()
    M.set(params)
    M.compute()
    
    # 4. Extract Spectra
    cl = M.raw_cl(2500)
    ell = cl['ell']
    factor = ell * (ell + 1) / (2 * np.pi) * (2.7255e6)**2
    
    tt = cl['tt'] * factor
    ee = cl['ee'] * factor
    te = cl['te'] * factor

    # 5. Visualization
    fig, axes = plt.subplots(3, 1, figsize=(10, 12))
    
    axes[0].plot(ell[2:], tt[2:], 'r-', label='Theory Prediction (TT)')
    axes[1].plot(ell[2:], ee[2:], 'b-', label='Theory Prediction (EE)')
    axes[2].plot(ell[2:], te[2:], 'g-', label='Theory Prediction (TE)')
    
    titles = ['CMB Temperature (TT)', 'E-Mode Polarization (EE)', 'Cross-Correlation (TE)']
    for i, ax in enumerate(axes):
        ax.set_title(titles[i])
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend()
        
    plt.tight_layout()
    save_path = os.path.join(output_dir, 'full_scientific_results.png')
    plt.savefig(save_path, dpi=300)
    
    print(f"[SUCCESS] High-fidelity run complete. Data plotted to {save_path}")
    
    # Cleanup
    M.struct_cleanup()
    M.empty()

if __name__ == "__main__":
    run_scientific_perturbation_test()